# Setup: Prerequisites for Korean LLM Evaluation

This notebook configures the required RBAC permissions and secrets to run `LMEvalJob` against an OAuth-protected KServe InferenceService on OpenShift AI.

## What is LMEvalJob?

LMEvalJob is a Custom Resource provided by the TrustyAI Operator that runs [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) as a Kubernetes Job. It handles dataset download, model inference, and result collection.

## Prerequisites

- OpenShift AI cluster with TrustyAI Operator installed
- A model deployed via KServe with `security.opendatahub.io/enable-auth: "true"`
- `oc` CLI logged in to the cluster
- Hugging Face token for gated model access

## Step 1: Set Your Configuration

Update these variables to match your environment:

In [ ]:
NAMESPACE = "hyo-project"        # Your OpenShift project
HF_TOKEN = "hf_xxxxx"           # Your Hugging Face token

## Step 2: Verify Cluster Access

In [ ]:
!oc whoami
!oc project {NAMESPACE}

## Step 3: Identify Your Model

Find the InferenceService name — this is also the `served_model_name` used by vLLM:

In [ ]:
!oc get inferenceservice -n {NAMESPACE}

## Step 4: Create RBAC Permissions

The LMEvalJob Pod uses the `default` ServiceAccount. When the InferenceService has OAuth enabled, the SA needs permission to `get` InferenceServices for the OAuth proxy to validate access.

In [ ]:
!oc apply -f samples/role.yaml -n {NAMESPACE}
!oc apply -f samples/rolebinding.yaml -n {NAMESPACE}

## Step 5: Create Hugging Face Token Secret

Required for downloading gated tokenizers (e.g., `google/gemma-2b`):

In [ ]:
!oc create secret generic hf-token \
    --from-literal=HF_TOKEN={HF_TOKEN} \
    -n {NAMESPACE} \
    --dry-run=client -o yaml | oc apply -f -

## Step 6: Create Service Account Token Secret

This long-lived SA token is injected as `OPENAI_API_KEY` to authenticate with the OAuth-protected model endpoint:

In [ ]:
!oc apply -f samples/sa-token-secret.yaml -n {NAMESPACE}

## Step 7: Verify Permissions

In [ ]:
!oc auth can-i get inferenceservices.serving.kserve.io \
    -n {NAMESPACE} \
    --as=system:serviceaccount:{NAMESPACE}:default

Expected output: `yes`

## Done!

You're now ready to run evaluations. Proceed to:
- **1_builtin_tasks/1_builtin_task_eval.ipynb** for a quick evaluation using built-in tasks
- **2_custom_tasks/1_custom_task_eval.ipynb** for full flexibility with Git-sourced tasks